# RAG Pipeline: Document Q&A for a Scottish Food & Drink SME

## Overview
This notebook implements a Retrieval-Augmented Generation (RAG) pipeline that allows
users to query a set of internal company documents using plain English questions.

## Business Problem
Staff at Highland Brew Co., a fictional Scottish craft brewery based in Inverness,
need to quickly retrieve information from internal compliance, supplier, and HR
documents without manually searching through multiple files. This pipeline enables
plain English querying across the full document library, returning precise,
grounded answers.

## Architecture
The pipeline has two distinct phases:

**Indexing phase** (run once upfront):
Load documents → Split into chunks → Embed chunks → Store in ChromaDB

**Query phase** (run on every user question):
Embed question → Retrieve similar chunks → Generate answer with LLM

## Tech Stack
- **LangChain** — framework for connecting LLMs to data and tools
- **ChromaDB** — local vector store for storing and searching embeddings
- **Sentence Transformers / HuggingFace** — local embedding model (all-MiniLM-L6-v2)
- **Groq API** — hosted inference for Llama 3.3 70b generative model
- **Google Colab** — cloud notebook environment

## Install packages



In [18]:
!pip install -q langchain langchain-google-genai langchain-community chromadb pypdf sentence-transformers groq langchain-text-splitters

## Imports and setup

In [ ]:
from google.colab import userdata
from groq import Groq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import urllib.request

# Initialise Groq client
groq_client = Groq(api_key=userdata.get('GROQ_API_KEY'))

# Initialise embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

## Step 1 — Load Documents

We load three plain text documents directly from GitHub using their raw URLs.

The three documents cover:
- **Compliance** — product specifications, allergens, quality control, traceability
- **Suppliers** — approved suppliers, procurement procedures, performance review
- **Staff** — health and safety, training requirements, leave and absence policy


Adapt accordingly for PDFs, Word documents, or database exports with appropriate document loaders.

In [22]:
import urllib.request

base_url = "https://raw.githubusercontent.com/Dazcam/applied-ai-portfolio/main/01-rag-pipeline/data/"

files = {
    "compliance": "highland_brew_compliance.txt",
    "suppliers": "highland_brew_suppliers.txt",
    "staff": "highland_brew_staff.txt"
}

documents = {}
for key, filename in files.items():
    url = base_url + filename
    with urllib.request.urlopen(url) as response:
        documents[key] = response.read().decode("utf-8")
    print(f"Loaded {filename} — {len(documents[key])} characters")

Loaded highland_brew_compliance.txt — 3852 characters
Loaded highland_brew_suppliers.txt — 2042 characters
Loaded highland_brew_staff.txt — 2157 characters


## Step 2 — Split Documents into Chunks

LLMs have a limit on how much text they can process at once (their context window).
More importantly, precise retrieval works better with smaller, focused chunks of text
than with entire documents. We split each document into overlapping chunks so that:

1. Each chunk is small enough to retrieve precisely
2. The overlap between chunks preserves context at boundaries — a sentence
   split across two chunks won't lose meaning in either

**Parameters:**
- `chunk_size=500` — maximum 500 characters per chunk
- `chunk_overlap=50` — 50 character overlap between consecutive chunks
- `separators=["\n\n", "\n", ".", " "]` — split on paragraph breaks first,
  then line breaks, then sentences — preserving natural document structure

**Tuning consideration:**
Chunk size and overlap are parameters you would tune based on document structure.
Highly structured documents like these work well with larger chunks. Dense prose
such as legal contracts might need smaller chunks for more precise retrieval.

Each chunk is tagged with its source document in the metadata, so the pipeline
always knows which document a retrieved chunk came from.

In [24]:
# Initialise the splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        # each chunk is max 500 characters
    chunk_overlap=50,      # chunks overlap by 50 characters to preserve context
    separators=["\n\n", "\n", ".", " "]  # split on paragraphs first, then lines, then sentences
)

# Split each document and tag each chunk with its source
all_chunks = []
for source, text in documents.items():
    chunks = splitter.create_documents(
        texts=[text],
        metadatas=[{"source": source}]  # tag so we know which doc each chunk came from
    )
    all_chunks.extend(chunks)
    print(f"{source}: {len(chunks)} chunks")

print(f"\nTotal chunks across all documents: {len(all_chunks)}")

compliance: 9 chunks
suppliers: 6 chunks
staff: 6 chunks

Total chunks across all documents: 21


## Step 3 — Embed Chunks and Store in ChromaDB

This is the core of the indexing phase. Each text chunk is converted into a
384-dimensional numerical vector (an embedding) that captures its semantic meaning.
These vectors are stored in ChromaDB alongside the original text.

**What is an embedding?**
An embedding is a list of numbers that represents the meaning of a piece of text
in a high-dimensional space. Texts with similar meanings produce vectors that point
in similar directions in that space. This allows us to search by meaning rather
than by keyword matching.

**Why a separate embedding model from the generative model?**
We use two different models in this pipeline for efficiency and cost reasons:

- **HuggingFace all-MiniLM-L6-v2** — a small, fast, specialist model whose only
  job is to produce embeddings. It runs locally for free. We use it for embedding
  because we potentially need to embed thousands of chunks and queries.
- **Groq / Llama 3.3 70b** — a large generative model capable of reading context
  and producing coherent answers. Used only once per query, after retrieval.

Using a large generative model for embedding would be slower, costly, and wasteful —
like using a surgeon to take a patient's temperature.

**What ChromaDB stores per chunk:**
1. The 384-dimensional embedding vector — used for similarity search
2. The original text — returned as a readable answer after retrieval
3. The source metadata — so we know which document the chunk came from

**Data security note:**
Because the embedding model runs locally, no document text is sent to any external
API during indexing. This is an important consideration for clients with sensitive
data — the entire indexing phase can remain within the client's own infrastructure.

In [ ]:
# Load the embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

# Test it on a single sentence so we can see what an embedding looks like
test_embedding = embeddings.embed_query("What is the shelf life of the cider?")

print(f"Embedding length: {len(test_embedding)} dimensions")
print(f"First 5 values: {test_embedding[:5]}")

In [26]:

# Create the vector store from our chunks
# This embeds all 21 chunks and stores them in ChromaDB
vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    collection_name="highland_brew"
)

print(f"Vector store created successfully")
print(f"Total vectors stored: {vectorstore._collection.count()}")

Vector store created successfully
Total vectors stored: 21


## Step 4 — Test Retrieval

Before involving the generative model, we test the retrieval step independently.
This is good practice — it lets us verify the vector search is working correctly
on its own, separate from the generation step.

**How retrieval works:**
1. The query is converted to a 384-dimensional embedding using the same model
2. ChromaDB computes the cosine similarity between the query vector and all 21
   stored chunk vectors
3. The k=3 most similar chunks are returned, along with their source metadata

**Cosine similarity** measures the angle between two vectors in high-dimensional
space. Vectors pointing in similar directions (small angle) have high cosine
similarity, indicating similar meaning — even if the exact words differ.

**Why k=3?**
We retrieve 3 chunks to give the generative model enough context to answer
accurately, without overwhelming it with irrelevant information. In a real system
with hundreds of documents, k would be tuned based on evaluation.

**Limitation:**
With only 21 chunks, the retriever occasionally returns weakly related chunks to
fill the k=3 slots. In a production system with a larger document library, more
relevant chunks would naturally displace these.


In [27]:
# Create a retriever from the vector store
# k=3 means return the 3 most relevant chunks for any query
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Test query
query = "What is the shelf life of the cider?"
retrieved_chunks = retriever.invoke(query)

print(f"Query: {query}")
print(f"\nTop {len(retrieved_chunks)} most relevant chunks:\n")
for i, chunk in enumerate(retrieved_chunks):
    print(f"--- Chunk {i+1} (source: {chunk.metadata['source']}) ---")
    print(chunk.page_content)
    print()

Query: What is the shelf life of the cider?

Top 3 most relevant chunks:

--- Chunk 1 (source: compliance) ---
3.3 Glen Light Lager
- ABV: 3.8%
- Ingredients: Water, Barley Malt, Wheat, Hops, Yeast
- Storage: Refrigerate after opening, consume within 3 days
- Shelf life: 9 months from production date
- Suitable for vegans: Yes

3.4 Caledonian Cider
- ABV: 4.5%
- Ingredients: Scottish Apples, Water, Sugar, Yeast
- Storage: Store in a cool dry place below 18 degrees Celsius
- Shelf life: 24 months from production date
- Suitable for vegans: Yes
- Gluten free: Yes

--- Chunk 2 (source: compliance) ---
3. PRODUCT SPECIFICATIONS
3.1 Highland Gold Ale
- ABV: 4.2%
- Ingredients: Water, Barley Malt, Hops, Yeast
- Storage: Store in a cool dry place below 18 degrees Celsius
- Shelf life: 12 months from production date
- Suitable for vegans: Yes

3.2 Inverness Dark Stout
- ABV: 5.1%
- Ingredients: Water, Barley Malt, Roasted Barley, Hops, Yeast
- Storage: Store in a cool dry place below 18 degree

## Step 5 — Full RAG Query Function

This function completes the query phase of the RAG pipeline. It combines retrieval
from ChromaDB with generation from the Llama 3.3 70b model hosted on Groq.

**The function does four things:**
1. Retrieves the 3 most relevant chunks from ChromaDB for the query
2. Formats those chunks into a context string, labelled by source document
3. Constructs a prompt that instructs the model to answer using only the
   provided context
4. Sends the prompt to Groq and returns the generated answer

**Grounding:**
The prompt explicitly instructs the model: "If the answer is not in the context,
say so clearly rather than making something up." This is called grounding —
constraining the model to the retrieved evidence rather than its general training
knowledge. It is a critical principle in responsible AI deployment, particularly
for business-facing tools where hallucinated answers could cause real harm.

**Data security consideration:**
At this stage, document chunks are sent to Groq's external API as part of the
prompt. For clients with sensitive data this may be a concern. Alternatives include:
- Running a local generative model via Ollama (no external API calls)
- Using an enterprise API such as Azure OpenAI where data handling is contractually
  guaranteed
- Anonymising sensitive content before retrieval

In [28]:
from groq import Groq

# Initialise Groq client
groq_client = Groq(api_key=userdata.get('GROQ_API_KEY'))

def ask_question(query):
    # Step 1: retrieve relevant chunks
    chunks = retriever.invoke(query)

    # Step 2: build context string from chunks
    context = "\n\n".join([
        f"[Source: {chunk.metadata['source']}]\n{chunk.page_content}"
        for chunk in chunks
    ])

    # Step 3: build prompt
    prompt = f"""You are a helpful assistant for Highland Brew Co., a Scottish craft brewery.
Answer the question below using only the context provided.
If the answer is not in the context, say so clearly rather than making something up.

Context:
{context}

Question: {query}

Answer:"""

    # Step 4: send to Groq
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

# Test it
answer = ask_question("What is the shelf life of the cider?")
print(answer)

The shelf life of the Caledonian Cider is 24 months from the production date.


## Steps 6a, 6b, 6c — Evaluation Queries

A pipeline that works on one query is not sufficient evidence that it works well.
We test three queries designed to evaluate different aspects of the system:

- **6a** — a query answered by the compliance document (emergency contacts)
- **6b** — a query answered by the staff document (annual leave) — tests
  cross-document retrieval
- **6c** — a query the documents cannot answer — tests whether grounding prevents
  hallucination

Structured evaluation like this is standard practice in applied AI delivery and
mirrors the kind of evidence needed to demonstrate reliability to a non-technical
stakeholder.

## Step 6a - Query: Emergency contact

In [29]:
answer = ask_question("Who should I contact in an emergency?")
print(answer)

In an emergency, you should contact the Emergency Recall Hotline at 0800 123 4567.


## Step 6b - Query: Annual leave entitlement

In [30]:
answer = ask_question("How much annual leave do staff get?")
print(answer)

Full-time staff are entitled to 28 days annual leave per year, including bank holidays. Part-time staff receive annual leave on a pro-rata basis, but the exact amount is not specified.


## Step 6c - Query: Out of scope question

In [31]:
answer = ask_question("What is the company's revenue last year?")
print(answer)

The company's revenue last year is not mentioned in the context.


## Step 7 — Demo

A clean end-to-end demonstration of the complete system. This cell can be run
independently once the indexing phase (Steps 1-3) has been completed.

In [32]:
# Demo — ask the system any question about Highland Brew Co. documents
questions = [
    "Is the Highland Gold Ale suitable for vegans?",
    "What happens if a supplier misses delivery targets?",
    "What PPE is required on the production floor?"
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask_question(q)}")
    print()

Q: Is the Highland Gold Ale suitable for vegans?
A: Yes, the Highland Gold Ale is suitable for vegans.

Q: What happens if a supplier misses delivery targets?
A: If a supplier fails to meet the on-time delivery rate target (above 95%) for two consecutive quarters, they are placed on review. If they remain on review for more than six months, they are removed from the approved list.

Q: What PPE is required on the production floor?
A: On the production floor, the required PPE includes: steel toe cap boots, hi-vis vest, hairnet, and gloves.



## Step 8 — Limitations and Potential Improvements

### What works well
- Precise retrieval of factual information from structured documents
- Cross-document retrieval — the pipeline searches all documents simultaneously
- Grounding — the model correctly declines to answer out of scope questions
- Clean separation of indexing and query phases
- No sensitive data sent externally during indexing (local embedding model)

### Current limitations
- **Small document library** — with only 21 chunks, weakly related chunks
  occasionally appear in the top 3 results. A larger document library would
  naturally improve retrieval precision.
- **Plain text only** — the current loader handles text files. Real SME documents
  are typically PDFs, Word files, or spreadsheets requiring additional parsing.
- **No conversation history** — each query is independent. The system cannot
  handle follow-up questions like "what about the stout?"
- **No source citation in answers** — the model knows which chunks it used but
  doesn't surface this in the answer by default.

### Potential improvements to explore
- **PDF and Word document support** — use LangChain document loaders for .pdf
  and .docx files to handle real SME document formats
- **Conversational memory** — add LangChain memory so the pipeline handles
  multi-turn conversations and follow-up questions
- **Source citation** — modify the prompt to instruct the model to cite which
  document each part of its answer comes from
- **Persistent vector store** — currently ChromaDB resets each session. Saving
  it to disk means documents only need to be indexed once
- **Chunk size experimentation** — evaluate retrieval quality with different
  chunk sizes and overlap values
- **Hybrid search** — combine semantic search with keyword search (BM25) for
  better precision on specific product codes or names
- **Evaluation framework** — implement a structured evaluation using a test set
  of questions with known answers to measure retrieval and answer quality
  systematically
- **Streamlit interface** — wrap the pipeline in a simple web interface so
  non-technical staff can query documents without touching a notebook